<h1 align="left">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_white.svg" width="500">
    <source media="(prefers-color-scheme: light)" srcset="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_black.svg" width="500">
    <img alt="Logo" src="https://autonomousvision.github.io/py123d/_static/123D_logo_transparent_black.svg" width="500">
  </picture>
  <h2 align="left">123D: Scene Filter Tutorial</h2>
</h1>

The `SceneFilter` is the central mechanism for selecting and configuring scenes from your converted datasets. It controls **which** logs to read, **what** metadata requirements to enforce, **how** to sample scenes temporally, and **what** post-processing to apply.

This tutorial walks through the different filter options step by step. Make sure you have downloaded demo logs as described in the [documentation](https://autonomousvision.github.io/py123d/installation/) before starting.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from py123d.api import SceneAPI, SceneFilter, get_filtered_scenes

## 1. Overview

The `SceneFilter` is a dataclass where all parameters default to `None` (no filtering). You only set what you need. The parameters cover:

- **Log selection** — pick datasets, splits, or specific logs
- **Metadata requirements** — map availability, locations, required sensor modalities
- **Temporal sampling** — scene duration, sampling rate, and spacing between scenes
- **Post-processing** — shuffle, limit, chunk, or apply custom filters

## 2. Default Filter: Load Everything

With an empty `SceneFilter`, all available logs are loaded. Each log produces a single scene spanning the full recording.

In [ ]:
scene_filter = SceneFilter()
scenes = get_filtered_scenes(scene_filter)

print(f"Total scenes: {len(scenes)}")
for scene in scenes:
    meta = scene.scene_metadata
    print(f"  {scene.split}/{scene.log_name}: {meta.num_future_iterations} iterations, {meta.future_duration_s:.1f}s")

## 3. Selecting Logs

These parameters control **which logs** to include:

- `datasets` — filter by dataset name (e.g. `["nuplan-mini"]`)
- `split_types` — filter by split type: `"train"`, `"val"`, or `"test"`
- `split_names` — filter by exact split name in the form `{dataset}_{split_type}`
- `log_names` — filter by specific log folder names

### 3.1 Filter by split type

Use `split_types` to restrict scenes to training, validation, or test splits.

In [ ]:
scene_filter = SceneFilter(split_types=["val"])
scenes = get_filtered_scenes(scene_filter)

print(f"Validation scenes: {len(scenes)}")
for scene in scenes:
    print(f"  Split: {scene.split}, Log: {scene.log_name}")

### 3.2 Filter by log name

If you know the specific log(s) you want, use `log_names` for direct selection.

In [ ]:
# Pick a specific log from our earlier results
all_scenes = get_filtered_scenes(SceneFilter())
example_log_name = all_scenes[0].log_name

scene_filter = SceneFilter(log_names=[example_log_name])
scenes = get_filtered_scenes(scene_filter)

print(f"Scenes from log '{example_log_name}': {len(scenes)}")

## 4. Metadata Requirements

These parameters filter logs based on their metadata, such as map properties or required sensor modalities.

- `has_map` — `True`: only logs with a map, `False`: only without, `None`: no filter
- `map_has_z` — `True`: only 3D maps (with elevation), `False`: only 2D maps
- `map_locations` / `log_locations` — filter by geographic location
- `required_scene_modalities` — require specific sensor modalities to be present

### 4.1 Filter by map availability

Many tasks (e.g. planning, routing) require a map. Use `has_map=True` to ensure all returned scenes have an associated map.

In [ ]:
scene_filter = SceneFilter(has_map=True)
scenes_with_map = get_filtered_scenes(scene_filter)

scene_filter = SceneFilter(has_map=False)
scenes_without_map = get_filtered_scenes(scene_filter)

print(f"Scenes with map:    {len(scenes_with_map)}")
print(f"Scenes without map: {len(scenes_without_map)}")

# Inspect map metadata of a scene with a map
if len(scenes_with_map) > 0:
    scene = scenes_with_map[0]
    map_meta = scene.get_map_metadata()
    print(f"\nMap metadata: {map_meta}")

### 4.2 Required scene modalities

The `required_scene_modalities` parameter ensures that specific sensors or data types are available (with no null values) across the entire scene. It supports three syntax forms:

- **Exact key**: `"camera.pcam_f0"` — this specific modality must be complete
- **Type `any`**: `"camera:any"` — at least one camera modality must be complete
- **Type `all`**: `"camera:all"` — every camera modality in the log must be complete

In [ ]:
# Require a front camera and ego state for every scene
scene_filter = SceneFilter(
    future_duration_s=1.0,
    required_scene_modalities=["camera.pcam_f0", "ego_state_se3"],
)
scenes = get_filtered_scenes(scene_filter)
print(f"Scenes with front camera + ego state: {len(scenes)}")

# Require at least one camera (any) and at least one lidar (any)
scene_filter = SceneFilter(
    future_duration_s=1.0,
    required_scene_modalities=["camera:any", "lidar:any"],
)
scenes = get_filtered_scenes(scene_filter)
print(f"Scenes with any camera + any lidar:   {len(scenes)}")

### 4.3 Inspect available modalities

To know which modality keys are valid for your dataset, you can inspect a scene's available modalities.

In [ ]:
scene = get_filtered_scenes(SceneFilter())[0]
all_metadatas = scene.get_all_modality_metadatas()

print("Available modality keys:")
for key, meta in all_metadatas.items():
    print(f"  {key:30s}  (type: {meta.modality_type.serialize()}, id: {meta.modality_id})")

## 5. Temporal Sampling

These parameters control **how** scenes are extracted from each log: their duration, sampling rate, and spacing.

### 5.1 Future and history duration

By default (no duration set), each log produces a single scene spanning the entire recording. Setting `future_duration_s` creates fixed-size sliding windows, generating multiple scenes per log. The `history_duration_s` controls how far back each scene can look.

In [ ]:
# No duration: one scene per log (entire recording)
scenes_full = get_filtered_scenes(SceneFilter())
print(f"No duration set:          {len(scenes_full)} scene(s)")
if len(scenes_full) > 0:
    meta = scenes_full[0].scene_metadata
    print(f"  Duration: {meta.future_duration_s:.1f}s, Iterations: {meta.num_future_iterations}")

# 2-second windows with 1-second history
scenes_windowed = get_filtered_scenes(SceneFilter(future_duration_s=2.0, history_duration_s=1.0))
print(f"\n2s future + 1s history:   {len(scenes_windowed)} scene(s)")
if len(scenes_windowed) > 0:
    meta = scenes_windowed[0].scene_metadata
    print(f"  Future duration: {meta.future_duration_s:.1f}s, Future iterations: {meta.num_future_iterations}")
    print(f"  History duration: {meta.history_duration_s:.1f}s, History iterations: {meta.num_history_iterations}")

### 5.2 Iteration stride and target duration

The **iteration stride** controls the sampling rate. A stride of `N` means every N-th raw frame becomes one iteration.

You can set it directly with `target_iteration_stride`, or specify a desired time step with `target_iteration_duration_s` (which auto-computes the stride from the inferred sampling frequency).

For example, on a 10 Hz log (0.1s per frame):
- `target_iteration_duration_s=0.5` yields stride = 5, giving an effective 2 Hz rate
- `target_iteration_stride=2` yields 5 Hz (every 2nd frame)

In [ ]:
# Native frequency (no stride)
scenes_native = get_filtered_scenes(SceneFilter(future_duration_s=2.0))

# 0.5s iteration duration (subsampled)
scenes_slow = get_filtered_scenes(SceneFilter(future_duration_s=2.0, target_iteration_duration_s=0.5))

if len(scenes_native) > 0 and len(scenes_slow) > 0:
    meta_native = scenes_native[0].scene_metadata
    meta_slow = scenes_slow[0].scene_metadata

    print("Native frequency:")
    print(f"  Iteration duration: {meta_native.iteration_duration_s:.3f}s")
    print(f"  Future iterations:  {meta_native.num_future_iterations}")

    print("\nWith target_iteration_duration_s=0.5:")
    print(f"  Iteration duration: {meta_slow.iteration_duration_s:.3f}s")
    print(f"  Future iterations:  {meta_slow.num_future_iterations}")

### 5.3 Controlling scene overlap with thresholds

When `future_duration_s` is set, multiple overlapping scenes are generated from each log. Use `timestamp_threshold_s` to enforce a minimum time gap between consecutive scene starts, reducing overlap.

Setting the threshold equal to the future duration creates non-overlapping scenes.

In [ ]:
duration_s = 2.0

# Overlapping scenes (default: one scene starts at every iteration)
scenes_overlap = get_filtered_scenes(SceneFilter(future_duration_s=duration_s))

# Non-overlapping: threshold = future duration
scenes_non_overlap = get_filtered_scenes(SceneFilter(future_duration_s=duration_s, timestamp_threshold_s=duration_s))

# Sparse: large gap between scenes
scenes_sparse = get_filtered_scenes(SceneFilter(future_duration_s=duration_s, timestamp_threshold_s=8.0))

print(f"Overlapping (no threshold):        {len(scenes_overlap)} scenes")
print(f"Non-overlapping (threshold={duration_s}s):  {len(scenes_non_overlap)} scenes")
print(f"Sparse (threshold=8s):             {len(scenes_sparse)} scenes")

### 5.4 Filter by scene UUID

If you have specific scene UUIDs (e.g. from a previous experiment), you can retrieve exactly those scenes. The UUID corresponds to the timestamp at iteration 0 of the scene.

In [ ]:
# Get some scenes and remember their UUIDs
scenes = get_filtered_scenes(SceneFilter(future_duration_s=1.0, max_num_scenes=5, shuffle=True))
uuids = [scene.scene_metadata.initial_uuid for scene in scenes]
print(f"Collected {len(uuids)} UUIDs:")
for uuid in uuids:
    print(f"  {uuid}")

# Retrieve exactly those scenes by UUID
scene_filter = SceneFilter(future_duration_s=1.0, scene_uuids=uuids)
scenes_by_uuid = get_filtered_scenes(scene_filter)
print(f"\nRetrieved {len(scenes_by_uuid)} scene(s) by UUID")

## 6. Post-processing

After scenes are generated, these options control the final output.

- `max_num_scenes` — cap the total number of scenes returned
- `shuffle` — randomly shuffle the scene order (applied last)
- `num_chunks` / `chunk_idx` — split scenes into chunks for distributed processing
- `custom_filter_fns` — apply arbitrary Python functions to filter scenes

### 6.1 Limit and shuffle

Use `max_num_scenes` to cap the number of returned scenes and `shuffle` to randomize the order. This is useful for quick experiments or random sampling.

In [ ]:
scene_filter = SceneFilter(
    future_duration_s=1.0,
    max_num_scenes=10,
    shuffle=True,
)
scenes = get_filtered_scenes(scene_filter)

print(f"Returned {len(scenes)} shuffled scene(s):")
for scene in scenes:
    print(f"  {scene.split} / {scene.log_name} — UUID: {scene.scene_metadata.initial_uuid[:8]}...")

### 6.2 Chunking for distributed processing

For parallel workloads, use `num_chunks` and `chunk_idx` to split scenes evenly. Each worker receives a disjoint subset.

In [ ]:
num_workers = 3
for worker_idx in range(num_workers):
    scene_filter = SceneFilter(
        future_duration_s=1.0,
        num_chunks=num_workers,
        chunk_idx=worker_idx,
    )
    chunk = get_filtered_scenes(scene_filter)
    print(f"Worker {worker_idx}: {len(chunk)} scenes")

### 6.3 Custom filter functions

For filters that cannot be expressed declaratively, pass Python callables via `custom_filter_fns`. Each function receives a `SceneAPI` and returns `True` to keep the scene.

> **Note:** Custom filter functions are not compatible with Hydra configuration overrides.

In [ ]:
def has_many_detections(scene: SceneAPI) -> bool:
    """Keep only scenes where the first frame has at least 5 box detections."""
    detections = scene.get_box_detections_se3_at_iteration(iteration=0)
    if detections is None:
        return False
    return len(detections) >= 5


scene_filter = SceneFilter(
    future_duration_s=1.0,
    custom_filter_fns=[has_many_detections],
    max_num_scenes=20,
)
scenes = get_filtered_scenes(scene_filter)

print(f"Scenes with >= 5 detections at t=0: {len(scenes)}")
if len(scenes) > 0:
    scene = scenes[0]
    detections = scene.get_box_detections_se3_at_iteration(iteration=0)
    print(f"  Example: {len(detections)} detections in first scene")

## 7. Putting It All Together

Here is a realistic example combining multiple filter options: select validation scenes with maps, at 2 Hz, in non-overlapping 1-second windows, limited to 10 scenes.

In [ ]:
scene_filter = SceneFilter(
    # Log selection
    split_types=["val"],
    # Metadata requirements
    has_map=True,
    required_scene_modalities=["ego_state_se3", "camera:any"],
    # Temporal sampling
    target_iteration_duration_s=0.5,  # 2 Hz
    future_duration_s=1.0,  # 1-second scenes
    history_duration_s=0.5,  # 1/2-second history
    timestamp_threshold_s=2.0,  # non-overlapping
    # Post-processing
    max_num_scenes=10,
    shuffle=True,
)
scenes = get_filtered_scenes(scene_filter)

print(f"Total scenes: {len(scenes)}\n")
for i, scene in enumerate(scenes):
    meta = scene.scene_metadata
    print(
        f"Scene {i}: split={scene.split}, "
        f"future={meta.num_future_iterations} iters ({meta.future_duration_s:.1f}s), "
        f"history={meta.num_history_iterations} iters ({meta.history_duration_s:.1f}s), "
        f"dt={meta.iteration_duration_s:.2f}s"
    )

### 7.1 Visualize a filtered scene

Let's visualize the timestamps of one of our filtered scenes to confirm the temporal configuration.

In [ ]:
from py123d.visualization.matplotlib.timestamps import plot_scene_timestamps

if len(scenes) > 0:
    scene: SceneAPI = np.random.choice(scenes)  # type: ignore
    print(f"Dataset: {scene.dataset}, Split: {scene.split}")
    print(f"Log: {scene.log_name}")

    fig, ax = plot_scene_timestamps(scene, include_history=True, time_unit="ms")
    plt.show()

## 8. Reference: All SceneFilter Parameters

| Parameter | Group | Description |
|---|---|---|
| `datasets` | Log selection | Dataset names to include |
| `split_types` | Log selection | Split types: `"train"`, `"val"`, `"test"` |
| `split_names` | Log selection | Exact split names (`{dataset}_{split}`) |
| `log_names` | Log selection | Specific log folder names |
| `has_map` | Metadata | Require/exclude map availability |
| `map_has_z` | Metadata | Require/exclude 3D map elevation |
| `map_locations` | Metadata | Filter by map location |
| `map_version` | Metadata | Filter by map metadata version |
| `log_locations` | Metadata | Filter by log location |
| `log_version` | Metadata | Filter by log metadata version |
| `scene_uuids` | Temporal | Specific scene UUIDs to retrieve |
| `target_iteration_duration_s` | Temporal | Desired time per iteration (auto-computes stride) |
| `target_iteration_stride` | Temporal | Skip every N raw frames |
| `future_duration_s` | Temporal | Scene future duration in seconds |
| `future_num_iterations` | Temporal | Scene future duration in iterations |
| `history_duration_s` | Temporal | Scene history duration in seconds |
| `history_num_iterations` | Temporal | Scene history duration in iterations |
| `timestamp_threshold_s` | Temporal | Min time gap between scene starts |
| `iteration_threshold` | Temporal | Min iteration gap between scene starts |
| `required_scene_modalities` | Metadata | Required modalities (exact keys or `type:any`/`type:all`) |
| `custom_filter_fns` | Post-processing | Custom Python filter functions |
| `num_chunks` | Post-processing | Number of chunks for distributed splits |
| `chunk_idx` | Post-processing | Chunk index to return (0-indexed) |
| `max_num_scenes` | Post-processing | Maximum scenes to return |
| `shuffle` | Post-processing | Shuffle scene order (default: `False`) |

For more details, see the [scene tutorial](01_scene_tutorial.ipynb) and [visualization tutorial](03_visualizations_tutorial.ipynb).